# Data Cleaning

## Approach

The original datasets are retained as raw baseline DataFrames and are not modified during the cleaning process. Separate working copies are used for all cleaning transformations.

Each approved data quality issue is addressed using the following workflow:

**Issue → Cleaning → Result → Explanation → Validation → Validation Result → Interpretation**

After all approved issues are resolved, a final validation is performed across the cleaned datasets before the analysis ready data is saved.

## Setup

In [1]:
# Install packages
from google.colab import drive
import pandas as pd
import duckdb
import numpy as np
import os

# Connect to Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Save file path
claims = "/content/drive/MyDrive/Resume 📃/Portfolio/Healthcare Claims Intelligence/Data/claims.csv"
members = "/content/drive/MyDrive/Resume 📃/Portfolio/Healthcare Claims Intelligence/Data/members.csv"
providers = "/content/drive/MyDrive/Resume 📃/Portfolio/Healthcare Claims Intelligence/Data/providers.csv"

# Load as a csv file
claims_raw = pd.read_csv(claims)
members_raw = pd.read_csv(members)
providers_raw = pd.read_csv(providers)

# Confirm raw dataset dimensions
print(f"Claims Raw:    {claims_raw.shape}")
print(f"Members Raw:   {members_raw.shape}")
print(f"Providers Raw: {providers_raw.shape}")

claims_clean = claims_raw.copy()
members_clean = members_raw.copy()
providers_clean = providers_raw.copy()

# Confirm working copies match the raw datasets
print(f"Claims Clean:    {claims_clean.shape}")
print(f"Members Clean:   {members_clean.shape}")
print(f"Providers Clean: {providers_clean.shape}")

# Convert date fields to datetime
claims_clean["service_date"] = pd.to_datetime(claims_clean["service_date"])
members_clean["date_of_birth"] = pd.to_datetime(members_clean["date_of_birth"])
members_clean["enrollment_date"] = pd.to_datetime(members_clean["enrollment_date"])
members_clean["termination_date"] = pd.to_datetime(members_clean["termination_date"])


Claims Raw:    (499996, 12)
Members Raw:   (50000, 8)
Providers Raw: (2500, 6)
Claims Clean:    (499996, 12)
Members Clean:   (50000, 8)
Providers Clean: (2500, 6)


## DQ-002: Enrollment Date Before Date of Birth

### Issue

During data profiling, 881 member records were identified where `enrollment_date` occurs before `date_of_birth`, representing 1.76% of the member population.

This violates the expected enrollment chronology because a member cannot be enrolled in a health plan before birth. If left unresolved, these records could affect eligibility analysis, "age at enrollment" calculations, and claims eligibility analysis.

**Cleaning Decision:**

When `enrollment_date` precedes `date_of_birth`, set `enrollment_date` equal to `date_of_birth`. The raw source data will remain unchanged.

During data profiling, 881 member records were identified where `enrollment_date` occurs before `date_of_birth`, representing 1.76% of the member population.

This violates the expected enrollment chronology because a member cannot be enrolled in a health plan before birth. If left unresolved, these records could affect eligibility analysis, "age at enrollment" calculations, and claims eligibility analysis.

**Approved Cleaning Decision:**

When `enrollment_date` precedes `date_of_birth`, set `enrollment_date` equal to `date_of_birth`. The raw source data will remain unchanged.

In [3]:
# Convert relevant fields to datetime
members_clean["date_of_birth"] = pd.to_datetime(members_clean["date_of_birth"])
members_clean["enrollment_date"] = pd.to_datetime(members_clean["enrollment_date"])

# Identify affected records (T/F boolean)
dq002_flag = members_clean["enrollment_date"] < members_clean["date_of_birth"]

print(f"Records identified for cleaning: {dq002_flag.sum():,}")

# Apply approved cleaning rule
members_clean.loc[dq002_flag, "enrollment_date"] = (
    members_clean.loc[dq002_flag, "date_of_birth"]
)

print(f"Records updated: {dq002_flag.sum():,}")

Records identified for cleaning: 881
Records updated: 881


### Validation

Validate that no member records remain where `enrollment_date` occurs before `date_of_birth`, and confirm that the raw source data remains unchanged.

In [4]:
# Validate enrollment chronology after cleaning
dq002_remaining = (
    members_clean["enrollment_date"] < members_clean["date_of_birth"]
).sum()

# Confirm the issue still exists in the untouched raw data
raw_dob = pd.to_datetime(members_raw["date_of_birth"])
raw_enrollment = pd.to_datetime(members_raw["enrollment_date"])

dq002_raw = (raw_enrollment < raw_dob).sum()

print(f"Invalid records remaining in clean data: {dq002_remaining:,}")
print(f"Invalid records retained in raw data:    {dq002_raw:,}")

Invalid records remaining in clean data: 0
Invalid records retained in raw data:    881


### Interpretation

The DQ-002 cleaning rule was successfully applied. All 881 member records with an `enrollment_date` before `date_of_birth` were corrected, and no invalid enrollment chronology remains in the cleaned member dataset.

The original raw dataset continues to contain the 881 identified records, confirming that the cleaning process modified only the working copy and preserved the source data.

## DQ-010: Claims Outside Member Enrollment Period

### Issue

During data profiling, 165,200 of 499,996 claim service lines were identified outside the member's enrollment period, representing 33.04% of all claim lines.

Of these, 141,273 claim lines occurred before the member's `enrollment_date`, while 23,927 occurred after `termination_date`. Claims outside valid enrollment periods could materially distort utilization, allowed cost, eligibility, member, and trend analyses.

**Cleaning Decision:**

After correcting invalid member enrollment dates in DQ-002, revalidate enrollment eligibility and exclude any remaining claim lines outside the valid enrollment period from the cleaned analytical dataset. Service dates will not be shifted because doing so would fabricate when healthcare services occurred.

In [5]:
dq010 = duckdb.sql("""
    SELECT
        c.*,
        m.enrollment_date,
        m.termination_date
    FROM claims_clean AS c
    LEFT JOIN members_clean AS m
        ON c.member_id = m.member_id
""").df()

dq010.head()

,claim_id,service_line_number,member_id,provider_id,service_date,procedure_code,procedure_description,diagnosis_code,diagnosis_description,place_of_service,service_category,allowed_amount,enrollment_date,termination_date
0,CLM00063913,2,MBR014774,PRV001174,2025-06-01,99202,"New patient office/outpatient visit, straightf...",N39.0,"Urinary tract infection, site not specified",Office,Primary Care,106.43,2022-04-12,NaT
1,CLM00063914,1,MBR005014,PRV002097,2024-11-08,99204,"New patient office/outpatient visit, moderate ...",A09,"Infectious gastroenteritis and colitis, unspec...",Office,Primary Care,215.88,2023-03-01,NaT
2,CLM00063915,1,MBR036127,PRV000488,2024-12-29,99212,"Established patient office/outpatient visit, s...",F90.9,"Attention-deficit hyperactivity disorder, unsp...",Office,Primary Care,81.71,2025-12-02,NaT
3,CLM00063915,2,MBR036127,PRV001176,2024-12-29,99244,"New/established outpatient consultation, moder...",F33.9,"Major depressive disorder, recurrent, unspecified",Office,Specialist Visit,210.97,2025-12-02,NaT
4,CLM00063916,1,MBR006947,PRV000344,2025-01-25,99204,"New patient office/outpatient visit, moderate ...",K52.9,"Noninfective gastroenteritis and colitis, unsp...",Office,Primary Care,195.20,2025-03-20,NaT


In [6]:
dq010_check = duckdb.sql("""
    SELECT
        COUNT(*) AS total_claims,
        SUM(CASE
            WHEN service_date < enrollment_date THEN 1
            ELSE 0
        END) AS before_enrollment,
        SUM(CASE
            WHEN termination_date IS NOT NULL
                 AND service_date > termination_date THEN 1
            ELSE 0
        END) AS after_termination
    FROM dq010
""").df()

dq010_check

,total_claims,before_enrollment,after_termination
0,499996,143408.0,23927.0


In [7]:
# Identify claims outside the valid enrollment period
dq010_flag = (
    (dq010["service_date"] < dq010["enrollment_date"])
    | (
        dq010["termination_date"].notna()
        & (dq010["service_date"] > dq010["termination_date"])
    )
)

print(f"Claims identified for exclusion: {dq010_flag.sum():,}")

# Keep only claims within the valid enrollment period
valid_claim_ids = dq010.loc[
    ~dq010_flag,
    ["claim_id", "service_line_number"]
]

claims_clean = claims_clean.merge(
    valid_claim_ids,
    on=["claim_id", "service_line_number"],
    how="inner"
)

print(f"Claims remaining after cleaning: {len(claims_clean):,}")

Claims identified for exclusion: 167,256
Claims remaining after cleaning: 332,740


### Validation

Validate that all remaining claim lines occur within the member's valid enrollment period, and confirm that the raw claims dataset remains unchanged.

In [8]:
# Join cleaned claims to corrected member enrollment dates
dq010_validation = duckdb.sql("""
    SELECT
        c.service_date,
        m.enrollment_date,
        m.termination_date
    FROM claims_clean AS c
    LEFT JOIN members_clean AS m
        ON c.member_id = m.member_id
""").df()

# Check for remaining enrollment violations
dq010_remaining = duckdb.sql("""
    SELECT
        COUNT(*) AS invalid_claims_remaining
    FROM dq010_validation
    WHERE service_date < enrollment_date
       OR (
            termination_date IS NOT NULL
            AND service_date > termination_date
       )
""").df()

print(
    f"Invalid claims remaining in clean data: "
    f"{dq010_remaining.loc[0, 'invalid_claims_remaining']:,}"
)

print(f"Claims retained in raw data:             {len(claims_raw):,}")
print(f"Claims remaining in clean data:          {len(claims_clean):,}")

Invalid claims remaining in clean data: 0
Claims retained in raw data:             499,996
Claims remaining in clean data:          332,740


### Interpretation

The DQ-010 cleaning rule was successfully applied. After incorporating the corrected enrollment dates from DQ-002, 167,256 claim lines outside valid member enrollment periods were excluded, leaving 332,740 valid claim lines for analysis.

No enrollment-period violations remain in the cleaned claims dataset. The original 499,996 claim lines remain unchanged in the raw dataset, confirming that the cleaning process modified only the working copy.

## DQ-011: Member and Provider State Mismatch

### Issue

During data profiling, 476,718 of 499,996 claim service lines connected members and providers located in different states, representing 95.34% of all claim lines.

Only 23,278 claim lines, or 4.66%, represented same-state service. This conflicts with the documented geography generation target of approximately 90% same-state service and could materially distort state-level utilization, provider access, network, and allowed-cost analyses.

**Cleaning Decision:**

Use controlled provider reassignment to restore the intended approximately 90% same-state service distribution while retaining realistic cross-state care. Selected affected claims will be reassigned to compatible same-state providers while preserving service category, place of service, provider compatibility, and network status where possible.

The raw source data will remain unchanged.

In [9]:
# Join cleaned claims to member and provider geography
dq011 = duckdb.sql("""
    SELECT
        c.*,
        m.state AS member_state,
        p.state AS provider_state,
        p.provider_type,
        p.specialty,
        p.network_status
    FROM claims_clean AS c
    LEFT JOIN members_clean AS m
        ON c.member_id = m.member_id
    LEFT JOIN providers_clean AS p
        ON c.provider_id = p.provider_id
""").df()

# Measure current geographic distribution
dq011_check = duckdb.sql("""
    SELECT
        CASE
            WHEN member_state = provider_state THEN 'Same State'
            ELSE 'Cross State'
        END AS geography_status,
        COUNT(*) AS claim_lines,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percent
    FROM dq011
    GROUP BY geography_status
    ORDER BY claim_lines DESC
""").df()

dq011_check

,geography_status,claim_lines,percent
0,Cross State,317188,95.33
1,Same State,15552,4.67


In [10]:
dq011_provider_check = duckdb.sql("""
    SELECT
        state,
        COUNT(*) AS providers,
        COUNT(DISTINCT provider_type) AS provider_types,
        COUNT(DISTINCT specialty) AS specialties
    FROM providers_clean
    GROUP BY state
    ORDER BY providers DESC
""").df()

dq011_provider_check

,state,providers,provider_types,specialties
0,CA,317,7,17
1,TX,235,7,17
2,FL,166,7,17
3,NY,160,7,17
4,OH,106,7,17
5,GA,95,7,17
6,NC,90,7,16
7,PA,85,7,16
8,IL,72,7,16
9,MI,72,7,16


In [11]:
dq011_state_check = duckdb.sql("""
    SELECT
        m.state AS member_state,
        COUNT(DISTINCT c.member_id) AS members_with_claims,
        COUNT(DISTINCT p.provider_id) AS available_providers
    FROM claims_clean AS c
    LEFT JOIN members_clean AS m
        ON c.member_id = m.member_id
    LEFT JOIN providers_clean AS p
        ON m.state = p.state
    GROUP BY m.state
    ORDER BY available_providers
""").df()

dq011_state_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,member_state,members_with_claims,available_providers
0,WY,84,4
1,ND,96,4
2,VT,73,4
3,SD,104,5
4,AK,79,5
5,MT,140,6
6,ME,155,7
7,DE,126,8
8,NH,173,10
9,RI,129,10


In [12]:
dq011_compatibility = duckdb.sql("""
    SELECT
        service_category,
        place_of_service,
        provider_type,
        specialty,
        COUNT(*) AS claim_lines
    FROM dq011
    GROUP BY
        service_category,
        place_of_service,
        provider_type,
        specialty
    ORDER BY
        service_category,
        place_of_service,
        claim_lines DESC
""").df()

dq011_compatibility

,service_category,place_of_service,provider_type,specialty,claim_lines
0,Emergency,Emergency Department,Hospital,General Acute Care,2857
1,Emergency,Emergency Department,Physician,Cardiology,1932
2,Emergency,Emergency Department,Physician,Orthopedics,1801
3,Emergency,Emergency Department,Imaging Center,Diagnostic Imaging,1788
4,Emergency,Emergency Department,Physician,Neurology,1774
...,...,...,...,...,...
261,Urgent Care,Urgent Care,Physician,OB/GYN,1071
262,Urgent Care,Urgent Care,Imaging Center,Diagnostic Imaging,840
263,Urgent Care,Urgent Care,Physician,Endocrinology,768
264,Urgent Care,Urgent Care,Clinic,Women's Health,718


In [13]:
# Define valid provider types by service category
dq011_rules = {
    "Primary Care": ["Physician", "Clinic"],
    "Specialist Visit": ["Physician", "Clinic"],
    "Preventive Care": ["Physician", "Clinic"],
    "Emergency": ["Hospital"],
    "Urgent Care": ["Urgent Care"],
    "Laboratory": ["Laboratory", "Hospital"],
    "Imaging": ["Imaging Center", "Hospital"],
    "Outpatient Procedure": [
        "Hospital",
        "Ambulatory Surgical Center",
        "Physician"
    ],
    "Inpatient": ["Hospital"],
    "Therapy": ["Clinic"]
}

In [14]:
# Create provider candidate pools
provider_candidates = []

for category, provider_types in dq011_rules.items():
    category_claims = dq011[
        dq011["service_category"] == category
    ][
        ["claim_id", "service_line_number", "member_state",
         "network_status"]
    ].copy()

    category_providers = providers_clean[
        providers_clean["provider_type"].isin(provider_types)
    ].copy()

    candidates = category_claims.merge(
        category_providers,
        left_on="member_state",
        right_on="state",
        how="left"
    )

    candidates["service_category"] = category

    provider_candidates.append(candidates)

provider_candidates = pd.concat(
    provider_candidates,
    ignore_index=True
)

dq011_availability = (
    provider_candidates
    .groupby(["claim_id", "service_line_number"])["provider_id"]
    .count()
)

print(
    f"Claim lines with no same-state compatible provider: "
    f"{(dq011_availability == 0).sum():,}"
)

print(
    f"Claim lines with at least one compatible provider:  "
    f"{(dq011_availability > 0).sum():,}"
)

Claim lines with no same-state compatible provider: 3,204
Claim lines with at least one compatible provider:  329,536


In [15]:
# Determine whether each claim has a same-state compatible provider
# with the same network status as the current provider
provider_candidates["network_match"] = (
    provider_candidates["network_status_x"]
    == provider_candidates["network_status_y"]
)

dq011_network_check = (
    provider_candidates
    .groupby(["claim_id", "service_line_number"])
    .agg(
        compatible_providers=("provider_id", "count"),
        same_network_providers=("network_match", "sum")
    )
    .reset_index()
)

print(
    f"Claims with same-state compatible provider: "
    f"{(dq011_network_check['compatible_providers'] > 0).sum():,}"
)

print(
    f"Claims with same-state compatible + same-network provider: "
    f"{(dq011_network_check['same_network_providers'] > 0).sum():,}"
)

print(
    f"Claims requiring network-status fallback: "
    f"{(
        (dq011_network_check['compatible_providers'] > 0)
        & (dq011_network_check['same_network_providers'] == 0)
    ).sum():,}"
)

Claims with same-state compatible provider: 329,536
Claims with same-state compatible + same-network provider: 320,712
Claims requiring network-status fallback: 8,824


In [16]:
# Define valid provider types by service category and place of service
dq011_pos_rules = {
    ("Primary Care", "Office"): ["Physician", "Clinic"],

    ("Specialist Visit", "Office"): ["Physician", "Clinic"],
    ("Specialist Visit", "Outpatient Hospital"): ["Physician", "Clinic"],

    ("Preventive Care", "Office"): ["Physician", "Clinic"],

    ("Emergency", "Emergency Department"): ["Hospital"],

    ("Urgent Care", "Urgent Care"): ["Urgent Care"],

    ("Laboratory", "Laboratory"): ["Laboratory", "Hospital"],
    ("Laboratory", "Outpatient Hospital"): ["Laboratory", "Hospital"],

    ("Imaging", "Imaging Center"): ["Imaging Center", "Hospital"],
    ("Imaging", "Outpatient Hospital"): ["Imaging Center", "Hospital"],

    ("Outpatient Procedure", "Outpatient Hospital"): [
        "Hospital",
        "Ambulatory Surgical Center",
        "Physician"
    ],
    ("Outpatient Procedure", "Ambulatory Surgical Center"): [
        "Hospital",
        "Ambulatory Surgical Center",
        "Physician"
    ],
    ("Outpatient Procedure", "Office"): [
        "Hospital",
        "Ambulatory Surgical Center",
        "Physician"
    ],

    ("Inpatient", "Inpatient Hospital"): ["Hospital"],

    ("Therapy", "Office"): ["Clinic"],
    ("Therapy", "Outpatient Hospital"): ["Clinic"]
}

In [17]:
dq011_pos_check = (
    claims_clean[
        ["service_category", "place_of_service"]
    ]
    .drop_duplicates()
    .copy()
)

dq011_pos_check["documented_rule"] = dq011_pos_check.apply(
    lambda row: (
        row["service_category"],
        row["place_of_service"]
    ) in dq011_pos_rules,
    axis=1
)

dq011_undocumented = dq011_pos_check[
    ~dq011_pos_check["documented_rule"]
].sort_values(
    ["service_category", "place_of_service"]
)

dq011_undocumented

,service_category,place_of_service,documented_rule


In [18]:
# Build fully compatible same-state provider candidate pools
dq011_final_candidates = []

for (category, pos), provider_types in dq011_pos_rules.items():

    # Claims matching this documented service category / POS rule
    category_claims = dq011[
        (dq011["service_category"] == category)
        & (dq011["place_of_service"] == pos)
    ][
        [
            "claim_id",
            "service_line_number",
            "member_state",
            "network_status"
        ]
    ].copy()

    # Providers with a compatible provider type
    category_providers = providers_clean[
        providers_clean["provider_type"].isin(provider_types)
    ].copy()

    # Require same state
    candidates = category_claims.merge(
        category_providers,
        left_on="member_state",
        right_on="state",
        how="left"
    )

    # Preserve network status
    candidates["network_match"] = (
        candidates["network_status_x"]
        == candidates["network_status_y"]
    )

    dq011_final_candidates.append(candidates)

dq011_final_candidates = pd.concat(
    dq011_final_candidates,
    ignore_index=True
)

In [19]:
dq011_final_check = (
    dq011_final_candidates
    .groupby(["claim_id", "service_line_number"])
    .agg(
        compatible_providers=("provider_id", "count"),
        same_network_providers=("network_match", "sum")
    )
    .reset_index()
)

print(
    f"Claims with fully compatible same-state provider: "
    f"{(dq011_final_check['compatible_providers'] > 0).sum():,}"
)

print(
    f"Claims with fully compatible same-state + same-network provider: "
    f"{(dq011_final_check['same_network_providers'] > 0).sum():,}"
)

print(
    f"Claims with no fully compatible same-state provider: "
    f"{(dq011_final_check['compatible_providers'] == 0).sum():,}"
)

Claims with fully compatible same-state provider: 329,536
Claims with fully compatible same-state + same-network provider: 320,712
Claims with no fully compatible same-state provider: 3,204


In [20]:
# Set approximate same-state target
dq011_target_rate = 0.895

total_claims = len(dq011)
current_same_state = (
    dq011["member_state"] == dq011["provider_state"]
).sum()

target_same_state = round(
    total_claims * dq011_target_rate
)

claims_to_reassign = (
    target_same_state - current_same_state
)

print(f"Total cleaned claims:           {total_claims:,}")
print(f"Current same-state claims:      {current_same_state:,}")
print(f"Target same-state claims:       {target_same_state:,}")
print(f"Cross-state claims to reassign: {claims_to_reassign:,}")

Total cleaned claims:           332,740
Current same-state claims:      15,552
Target same-state claims:       297,802
Cross-state claims to reassign: 282,250


In [21]:
# Identify current cross-state claims
dq011_cross_state = dq011[
    dq011["member_state"] != dq011["provider_state"]
][
    ["claim_id", "service_line_number"]
].copy()

# Find cross-state claims with at least one fully compatible
# same-state and same-network provider
dq011_eligible = (
    dq011_final_check[
        dq011_final_check["same_network_providers"] > 0
    ]
    .merge(
        dq011_cross_state,
        on=["claim_id", "service_line_number"],
        how="inner"
    )
)

print(
    f"Cross-state claims eligible for reassignment: "
    f"{len(dq011_eligible):,}"
)

print(
    f"Cross-state claims needed for target:         "
    f"{claims_to_reassign:,}"
)

print(
    f"Eligible claims remaining after target:       "
    f"{len(dq011_eligible) - claims_to_reassign:,}"
)

Cross-state claims eligible for reassignment: 305,404
Cross-state claims needed for target:         282,250
Eligible claims remaining after target:       23,154


In [22]:
# Randomly select eligible cross-state claims for reassignment
dq011_selected = dq011_eligible.sample(
    n=claims_to_reassign,
    random_state=42
)[
    ["claim_id", "service_line_number"]
].copy()

print(f"Claims selected for reassignment: {len(dq011_selected):,}")

Claims selected for reassignment: 282,250


In [23]:
# Restrict candidate pool to selected claims
# and require matching network status
dq011_assignment_pool = (
    dq011_final_candidates[
        dq011_final_candidates["network_match"]
    ]
    .merge(
        dq011_selected,
        on=["claim_id", "service_line_number"],
        how="inner"
    )
)

In [24]:
# Randomly select one compatible provider for each claim
dq011_assignments = (
    dq011_assignment_pool
    .sample(
        frac=1,
        random_state=42
    )
    .drop_duplicates(
        subset=["claim_id", "service_line_number"]
    )
    [
        ["claim_id", "service_line_number", "provider_id"]
    ]
    .rename(
        columns={"provider_id": "new_provider_id"}
    )
)

print(
    f"Provider assignments created: "
    f"{len(dq011_assignments):,}"
)

Provider assignments created: 282,250


In [25]:
# Add new provider assignments to cleaned claims
claims_clean = claims_clean.merge(
    dq011_assignments,
    on=["claim_id", "service_line_number"],
    how="left"
)

# Replace provider_id for selected claims
claims_clean["provider_id"] = (
    claims_clean["new_provider_id"]
    .fillna(claims_clean["provider_id"])
)

# Remove temporary assignment field
claims_clean = claims_clean.drop(
    columns=["new_provider_id"]
)

print(
    f"Claims remaining after provider reassignment: "
    f"{len(claims_clean):,}"
)

print(
    f"Claims reassigned to new providers: "
    f"{len(dq011_assignments):,}"
)

Claims remaining after provider reassignment: 332,740
Claims reassigned to new providers: 282,250


### Validation

Validate the provider reassignment by confirming the resulting same state service distribution, provider referential integrity, and preservation of network status and documented provider compatibility.

In [26]:
dq011_validation = duckdb.sql("""
    SELECT
        c.claim_id,
        c.service_line_number,
        c.member_id,
        c.provider_id,
        c.service_category,
        c.place_of_service,
        m.state AS member_state,
        p.state AS provider_state,
        p.provider_type,
        p.specialty,
        p.network_status
    FROM claims_clean AS c
    LEFT JOIN members_clean AS m
        ON c.member_id = m.member_id
    LEFT JOIN providers_clean AS p
        ON c.provider_id = p.provider_id
""").df()

dq011_validation["geography_status"] = (
    dq011_validation["member_state"]
    == dq011_validation["provider_state"]
).map({
    True: "Same State",
    False: "Cross State"
})

dq011_geography_validation = (
    dq011_validation["geography_status"]
    .value_counts()
    .rename_axis("geography_status")
    .reset_index(name="claim_lines")
)

dq011_geography_validation["percent"] = (
    dq011_geography_validation["claim_lines"]
    / len(dq011_validation)
    * 100
).round(2)

dq011_geography_validation

,geography_status,claim_lines,percent
0,Same State,297802,89.5
1,Cross State,34938,10.5


In [27]:
# Validate provider referential integrity
missing_providers = dq011_validation["provider_state"].isna().sum()

# Validate service category + POS + provider type compatibility
dq011_validation["valid_provider_rule"] = dq011_validation.apply(
    lambda row: row["provider_type"] in dq011_pos_rules.get(
        (row["service_category"], row["place_of_service"]),
        []
    ),
    axis=1
)

invalid_provider_compatibility = (
    ~dq011_validation["valid_provider_rule"]
).sum()

print(
    f"Claims with missing provider reference: "
    f"{missing_providers:,}"
)

print(
    f"Claims violating documented provider compatibility: "
    f"{invalid_provider_compatibility:,}"
)

Claims with missing provider reference: 0
Claims violating documented provider compatibility: 31,686


In [28]:
# Identify whether compatibility violations were reassigned by DQ-011
dq011_validation_check = dq011_validation.merge(
    dq011_assignments[
        ["claim_id", "service_line_number"]
    ].assign(reassigned=True),
    on=["claim_id", "service_line_number"],
    how="left"
)

dq011_validation_check["reassigned"] = (
    dq011_validation_check["reassigned"]
    .fillna(False)
    .astype(bool)
)

dq011_invalid_source = (
    dq011_validation_check[
        ~dq011_validation_check["valid_provider_rule"]
    ]
    .groupby("reassigned")
    .size()
    .reset_index(name="invalid_claim_lines")
)

dq011_invalid_source

/tmp/ipykernel_2307/2203630890.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,reassigned,invalid_claim_lines
0,False,31686


In [29]:
dq011_invalid_detail = (
    dq011_validation_check[
        ~dq011_validation_check["valid_provider_rule"]
    ]
    .groupby(
        [
            "service_category",
            "place_of_service",
            "provider_type"
        ]
    )
    .size()
    .reset_index(name="claim_lines")
    .sort_values(
        "claim_lines",
        ascending=False
    )
)

dq011_invalid_detail

,service_category,place_of_service,provider_type,claim_lines
70,Urgent Care,Urgent Care,Physician,3723
4,Emergency,Emergency Department,Physician,3184
30,Laboratory,Outpatient Hospital,Physician,2339
14,Imaging,Outpatient Hospital,Physician,2001
63,Therapy,Office,Physician,1562
...,...,...,...,...
43,Outpatient Procedure,Outpatient Hospital,Urgent Care,17
34,Outpatient Procedure,Ambulatory Surgical Center,Laboratory,16
35,Outpatient Procedure,Ambulatory Surgical Center,Urgent Care,15
42,Outpatient Procedure,Outpatient Hospital,Laboratory,10


### Interpretation

The provider reassignment successfully corrected the geographic imbalance identified during profiling. The initial cleaned dataset contained only 4.67% same state service, compared with the documented target of approximately 90%.

After controlled provider reassignment, the same state rate reached 89.50%. Subsequent correction of provider compatibility issues under DQ-014 resulted in a final same state rate of 93.25%, which remains reasonably consistent with the approximate geography target while retaining 6.75% cross-state care.

The reassignment process preserved claim volume and did not introduce new provider compatibility violations. DQ-011 is considered resolved.

## DQ-014: Provider Service Compatibility Violation

### Issue

During validation of DQ-011, 31,686 claim service lines were found to be assigned to provider types that are incompatible with their documented service category and place of service.

These violations existed outside the claims reassigned during DQ-011 and were not introduced by the geographic correction.

**Cleaning Decision:**

Reassign affected claims to providers with a compatible provider type while preserving member state and network status where possible.

In [30]:
# Isolate the incompatible claims
dq014_invalid = dq011_validation_check[
    ~dq011_validation_check["valid_provider_rule"]
][
    [
        "claim_id",
        "service_line_number",
        "service_category",
        "place_of_service",
        "member_state",
        "network_status"
    ]
].copy()

dq014_candidates = []

for (category, pos), provider_types in dq011_pos_rules.items():

    affected_claims = dq014_invalid[
        (dq014_invalid["service_category"] == category)
        & (dq014_invalid["place_of_service"] == pos)
    ].copy()

    compatible_providers = providers_clean[
        providers_clean["provider_type"].isin(provider_types)
    ].copy()

    candidates = affected_claims.merge(
        compatible_providers,
        left_on="member_state",
        right_on="state",
        how="left",
        suffixes=("_claim", "_provider")
    )

    candidates["network_match"] = (
        candidates["network_status_claim"]
        == candidates["network_status_provider"]
    )

    dq014_candidates.append(candidates)

dq014_candidates = pd.concat(
    dq014_candidates,
    ignore_index=True
)

dq014_feasibility = (
    dq014_candidates
    .groupby(["claim_id", "service_line_number"])
    .agg(
        compatible_same_state=("provider_id", "count"),
        compatible_same_state_network=("network_match", "sum")
    )
    .reset_index()
)

print(f"Incompatible claims: {len(dq014_invalid):,}")

print(
    "Compatible same-state provider available: "
    f"{(dq014_feasibility['compatible_same_state'] > 0).sum():,}"
)

print(
    "Compatible same-state + same-network provider available: "
    f"{(dq014_feasibility['compatible_same_state_network'] > 0).sum():,}"
)

Incompatible claims: 31,686
Compatible same-state provider available: 28,802
Compatible same-state + same-network provider available: 21,274


In [31]:
dq014_national_candidates = []

for (category, pos), provider_types in dq011_pos_rules.items():

    affected_claims = dq014_invalid[
        (dq014_invalid["service_category"] == category)
        & (dq014_invalid["place_of_service"] == pos)
    ].copy()

    compatible_providers = providers_clean[
        providers_clean["provider_type"].isin(provider_types)
    ].copy()

    candidates = affected_claims.merge(
        compatible_providers,
        left_on="network_status",
        right_on="network_status",
        how="left",
        suffixes=("_claim", "_provider")
    )

    dq014_national_candidates.append(candidates)

dq014_national_candidates = pd.concat(
    dq014_national_candidates,
    ignore_index=True
)

dq014_national_availability = (
    dq014_national_candidates
    .groupby(["claim_id", "service_line_number"])["provider_id"]
    .count()
)

print(
    "Claims with compatible same-network provider available: "
    f"{(dq014_national_availability > 0).sum():,}"
)

print(
    "Claims with no compatible same-network provider available: "
    f"{(dq014_national_availability == 0).sum():,}"
)


Claims with compatible same-network provider available: 31,686
Claims with no compatible same-network provider available: 0


In [32]:
# Mark whether each compatible provider is in the member's state
dq014_national_candidates["same_state"] = (
    dq014_national_candidates["member_state"]
    == dq014_national_candidates["state"]
)

# Randomize candidates reproducibly
dq014_assignment_pool = (
    dq014_national_candidates
    .sample(
        frac=1,
        random_state=42
    )
    .sort_values(
        "same_state",
        ascending=False
    )
)

# Select one provider per incompatible claim
# Same-state candidates are prioritized
dq014_assignments = (
    dq014_assignment_pool
    .drop_duplicates(
        subset=["claim_id", "service_line_number"]
    )
    [
        [
            "claim_id",
            "service_line_number",
            "provider_id",
            "same_state"
        ]
    ]
    .rename(
        columns={"provider_id": "new_provider_id"}
    )
)

print(
    f"Provider assignments created: "
    f"{len(dq014_assignments):,}"
)

print(
    f"Same-state assignments: "
    f"{dq014_assignments['same_state'].sum():,}"
)

print(
    f"Cross-state fallback assignments: "
    f"{(~dq014_assignments['same_state']).sum():,}"
)

Provider assignments created: 31,686
Same-state assignments: 21,274
Cross-state fallback assignments: 10,412


In [33]:
# Keep only fields needed for the provider update
dq014_provider_updates = dq014_assignments[
    ["claim_id", "service_line_number", "new_provider_id"]
].copy()

# Merge replacement providers onto cleaned claims
claims_clean = claims_clean.merge(
    dq014_provider_updates,
    on=["claim_id", "service_line_number"],
    how="left"
)

# Replace provider_id only for DQ-014 affected claims
claims_clean["provider_id"] = (
    claims_clean["new_provider_id"]
    .fillna(claims_clean["provider_id"])
)

# Remove temporary field
claims_clean = claims_clean.drop(
    columns=["new_provider_id"]
)

print(f"Claims remaining: {len(claims_clean):,}")
print(f"Claims corrected for DQ-014: {len(dq014_provider_updates):,}")

Claims remaining: 332,740
Claims corrected for DQ-014: 31,686


### Validation

In [34]:
# Rebuild claim-provider relationships after DQ-014
dq014_validation = duckdb.sql("""
    SELECT
        c.claim_id,
        c.service_line_number,
        c.service_category,
        c.place_of_service,
        m.state AS member_state,
        p.state AS provider_state,
        p.provider_type,
        p.network_status
    FROM claims_clean AS c
    LEFT JOIN members_clean AS m
        ON c.member_id = m.member_id
    LEFT JOIN providers_clean AS p
        ON c.provider_id = p.provider_id
""").df()

# Validate provider compatibility
dq014_validation["valid_provider_rule"] = dq014_validation.apply(
    lambda row: row["provider_type"] in dq011_pos_rules.get(
        (row["service_category"], row["place_of_service"]),
        []
    ),
    axis=1
)

invalid_compatibility = (
    ~dq014_validation["valid_provider_rule"]
).sum()

# Validate geography
same_state = (
    dq014_validation["member_state"]
    == dq014_validation["provider_state"]
).sum()

cross_state = len(dq014_validation) - same_state
same_state_rate = same_state / len(dq014_validation) * 100

print(
    f"Provider compatibility violations remaining: "
    f"{invalid_compatibility:,}"
)

print(f"Same-state claims:  {same_state:,}")
print(f"Cross-state claims: {cross_state:,}")
print(f"Same-state rate:    {same_state_rate:.2f}%")

Provider compatibility violations remaining: 0
Same-state claims:  310,285
Cross-state claims: 22,455
Same-state rate:    93.25%


### Interpretation

Validation confirmed that all 31,686 provider compatibility violations identified during DQ-011 validation were resolved. After provider reassignment, zero claim lines remained assigned to provider types that violated the documented service category and place-of-service compatibility rules.

All 332,740 claim lines were retained, network status was preserved during reassignment, and the resulting 93.25% same-state service rate remained reasonably consistent with the documented approximate geography target.

DQ-014 is considered resolved.

## DQ-012: Missing Network Effect in Allowed Amount

### Issue

During data profiling, normalized allowed amounts were nearly identical between in-network and out-of-network claims. The average normalized allowed amount was 1.034999 for in-network claims and 1.034916 for out-of-network claims, indicating that the intended network cost adjustment was not reflected in the generated claims data.

This limits the ability to analyze differences in healthcare cost by provider network status.

**Cleaning Decision:**

Recalculate allowed amounts after provider and geography corrections using the documented synthetic data generation rules. The recalculation will incorporate the procedure baseline allowed amount, allowed amount variation, network adjustment, and geography adjustment using the final provider assignment.

The raw source data will remain unchanged.

In [35]:
procedure_reference = "/content/drive/MyDrive/Resume 📃/Portfolio/Healthcare Claims Intelligence/Data/procedure_reference.csv"

procedure_reference_raw = pd.read_csv(procedure_reference)

display(procedure_reference_raw.head())
print(f"Procedure Reference: {procedure_reference_raw.shape}")

,procedure_code,procedure_description,service_category,baseline_allowed_amount
0,99202,"New patient office/outpatient visit, straightf...",Primary Care,85
1,99203,"New patient office/outpatient visit, low compl...",Primary Care,125
2,99204,"New patient office/outpatient visit, moderate ...",Primary Care,185
3,99205,"New patient office/outpatient visit, high comp...",Primary Care,245
4,99212,"Established patient office/outpatient visit, s...",Primary Care,65


Procedure Reference: (75, 4)


In [36]:
# Add procedure baseline to cleaned claims
claims_clean = claims_clean.merge(
    procedure_reference_raw[
        ["procedure_code", "baseline_allowed_amount"]
    ],
    on="procedure_code",
    how="left"
)

# Preserve current allowed amount for validation
claims_clean["allowed_amount_original"] = (
    claims_clean["allowed_amount"]
)

print(
    "Claims missing procedure baseline: "
    f"{claims_clean['baseline_allowed_amount'].isna().sum():,}"
)

Claims missing procedure baseline: 0


In [37]:
# Add final provider network status to claims
claims_clean = claims_clean.merge(
    providers_clean[
        ["provider_id", "network_status"]
    ],
    on="provider_id",
    how="left"
)

print(
    "Claims missing network status: "
    f"{claims_clean['network_status'].isna().sum():,}"
)

print(
    claims_clean["network_status"]
    .value_counts(dropna=False)
)

Claims missing network status: 0
network_status
In-Network        265419
Out-of-Network     67321
Name: count, dtype: int64


In [38]:
# Set seed so results are reproducible
rng = np.random.default_rng(42)

# Generate natural cost variation around procedure baseline
variation = rng.normal(
    loc=1.0,
    scale=0.08,
    size=len(claims_clean)
)

# Limit natural variation to approximately ±20%
variation = np.clip(variation, 0.80, 1.20)

# Calculate allowed amount from procedure baseline
claims_clean["allowed_amount"] = (
    claims_clean["baseline_allowed_amount"] * variation
)

# Apply approximately 20% upward adjustment for out-of-network services
out_of_network = (
    claims_clean["network_status"] == "Out-of-Network"
)

claims_clean.loc[
    out_of_network,
    "allowed_amount"
] *= 1.20

# Round to currency precision
claims_clean["allowed_amount"] = (
    claims_clean["allowed_amount"].round(2)
)

print(f"Claims recalculated: {len(claims_clean):,}")
print(
    "Allowed amounts <= 0: "
    f"{(claims_clean['allowed_amount'] <= 0).sum():,}"
)

Claims recalculated: 332,740
Allowed amounts <= 0: 0


### Validation

Validate the recalculated allowed amounts by comparing normalized allowed amounts between in-network and out-of-network claims. Normalization controls for differences in procedure baseline costs and allows the intended network cost effect to be evaluated directly.

In [39]:
# Calculate normalized allowed amount
claims_clean["normalized_allowed_amount"] = (
    claims_clean["allowed_amount"]
    / claims_clean["baseline_allowed_amount"]
)

# Compare normalized cost by network status
network_validation = (
    claims_clean
    .groupby("network_status")
    .agg(
        claim_lines=("claim_id", "size"),
        avg_allowed_amount=("allowed_amount", "mean"),
        avg_normalized_allowed=("normalized_allowed_amount", "mean"),
        min_normalized_allowed=("normalized_allowed_amount", "min"),
        max_normalized_allowed=("normalized_allowed_amount", "max")
    )
    .round(4)
)

display(network_validation)

# Calculate relative network effect
in_network_avg = network_validation.loc[
    "In-Network", "avg_normalized_allowed"
]

out_network_avg = network_validation.loc[
    "Out-of-Network", "avg_normalized_allowed"
]

network_effect = (
    (out_network_avg / in_network_avg) - 1
) * 100

print(
    f"Out-of-network normalized cost increase: "
    f"{network_effect:.2f}%"
)

print(
    f"Allowed amounts <= 0: "
    f"{(claims_clean['allowed_amount'] <= 0).sum():,}"
)

,claim_lines,avg_allowed_amount,avg_normalized_allowed,min_normalized_allowed,max_normalized_allowed
network_status,,,,,
In-Network,265419,251.8155,1.0,0.80,1.20
Out-of-Network,67321,311.2320,1.2,0.96,1.44


Out-of-network normalized cost increase: 20.00%
Allowed amounts <= 0: 0


### Interpretation

Validation confirmed that the recalculated allowed amounts now reflect the documented network cost effect. After controlling for differences in procedure baseline costs, the average normalized allowed amount was 1.00 for in-network claims and 1.20 for out-of-network claims, representing a 20.00% average increase for out-of-network services.

Natural cost variation was retained around the procedure-specific baseline, resulting in overlapping allowed amount ranges between network groups. All recalculated allowed amounts remained greater than zero.

## DQ-013: Artificial Temporal Utilization Growth

### Issue

During data profiling, claim utilization increased substantially from 2021 through 2025 beyond what could be explained by changes in member enrollment. Claim lines per claiming member increased from approximately 3.02 in 2021 to 7.51 in 2025, while monthly claim lines per 1,000 enrolled members increased from approximately 101 in January 2021 to 1,176 in December 2025.

The synthetic data generation rules do not define sustained year-over-year utilization growth of this magnitude as an intended characteristic of the dataset. Leaving this pattern unchanged could create misleading utilization and cost trends in downstream analysis.

**Cleaning Decision:**

Correct the temporal distribution of claims while preserving valid member enrollment periods, differences in member utilization, procedure and diagnosis relationships, service-category composition, allowed amounts, and realistic month-to-month variation. The correction will not force identical utilization across months or years.

In [40]:
# Measure current cleaned claim volume by year
dq013_current = (
    claims_clean
    .assign(service_year=claims_clean["service_date"].dt.year)
    .groupby("service_year")
    .agg(
        claim_lines=("claim_id", "size"),
        claiming_members=("member_id", "nunique")
    )
)

dq013_current["lines_per_claiming_member"] = (
    dq013_current["claim_lines"]
    / dq013_current["claiming_members"]
)

display(dq013_current.round(2))

,claim_lines,claiming_members,lines_per_claiming_member
service_year,,,
2021,681,259,2.63
2022,5100,1663,3.07
2023,20856,5862,3.56
2024,64437,15278,4.22
2025,241666,34417,7.02


In [41]:
# Calculate enrolled members by year
enrollment_by_year = []

for year in range(2021, 2026):
    year_start = pd.Timestamp(f"{year}-01-01")
    year_end = pd.Timestamp(f"{year}-12-31")

    enrolled = (
        (members_clean["enrollment_date"] <= year_end)
        & (
            members_clean["termination_date"].isna()
            | (members_clean["termination_date"] >= year_start)
        )
    )

    enrollment_by_year.append({
        "service_year": year,
        "enrolled_members": enrolled.sum()
    })

enrollment_by_year = pd.DataFrame(
    enrollment_by_year
).set_index("service_year")

dq013_yearly = dq013_current.join(
    enrollment_by_year
)

dq013_yearly["claim_lines_per_1000_enrolled"] = (
    dq013_yearly["claim_lines"]
    / dq013_yearly["enrolled_members"]
    * 1000
)

display(dq013_yearly.round(2))

,claim_lines,claiming_members,lines_per_claiming_member,enrolled_members,claim_lines_per_1000_enrolled
service_year,,,,,
2021,681,259,2.63,5843,116.55
2022,5100,1663,3.07,13160,387.54
2023,20856,5862,3.56,22423,930.12
2024,64437,15278,4.22,34008,1894.76
2025,241666,34417,7.02,48149,5019.13


In [42]:
# Join member enrollment windows to claims
dq013 = claims_clean.merge(
    members_clean[
        [
            "member_id",
            "enrollment_date",
            "termination_date"
        ]
    ],
    on="member_id",
    how="left"
)

# Limit open enrollment windows to the end of the analysis period
analysis_end = pd.Timestamp("2025-12-31")

dq013["valid_end_date"] = (
    dq013["termination_date"]
    .fillna(analysis_end)
    .clip(upper=analysis_end)
)

# Confirm every claim has a valid enrollment window
invalid_windows = (
    dq013["enrollment_date"]
    > dq013["valid_end_date"]
).sum()

print(f"Claims with invalid enrollment windows: {invalid_windows:,}")

display(
    dq013[
        [
            "member_id",
            "service_date",
            "enrollment_date",
            "valid_end_date"
        ]
    ].head()
)

Claims with invalid enrollment windows: 0


,member_id,service_date,enrollment_date,valid_end_date
0,MBR001379,2023-10-15,2021-12-04,2025-12-31
1,MBR017080,2025-07-29,2024-09-10,2025-12-31
2,MBR002416,2025-08-26,2023-07-22,2025-12-31
3,MBR037993,2025-11-26,2022-05-14,2025-12-31
4,MBR037993,2025-11-26,2022-05-14,2025-12-31


In [43]:
# Set seed for reproducible date simulation
rng = np.random.default_rng(42)

# Calculate number of valid days available for each claim
valid_days = (
    dq013["valid_end_date"]
    - dq013["enrollment_date"]
).dt.days

# Generate a random day offset within each member's enrollment window
random_offsets = (
    rng.random(len(dq013))
    * (valid_days + 1)
).astype(int)

# Create simulated service dates
dq013["simulated_service_date"] = (
    dq013["enrollment_date"]
    + pd.to_timedelta(random_offsets, unit="D")
)

# Summarize simulated distribution by year
dq013_simulation = (
    dq013
    .assign(
        service_year=dq013["simulated_service_date"].dt.year
    )
    .groupby("service_year")
    .agg(
        claim_lines=("claim_id", "size"),
        claiming_members=("member_id", "nunique")
    )
)

dq013_simulation["lines_per_claiming_member"] = (
    dq013_simulation["claim_lines"]
    / dq013_simulation["claiming_members"]
)

dq013_simulation = dq013_simulation.join(
    enrollment_by_year
)

dq013_simulation["claim_lines_per_1000_enrolled"] = (
    dq013_simulation["claim_lines"]
    / dq013_simulation["enrolled_members"]
    * 1000
)

display(dq013_simulation.round(2))

print(
    f"Simulated claims: "
    f"{dq013_simulation['claim_lines'].sum():,}"
)

,claim_lines,claiming_members,lines_per_claiming_member,enrolled_members,claim_lines_per_1000_enrolled
service_year,,,,,
2021,5677,2760,2.06,5843,971.59
2022,21625,8106,2.67,13160,1643.24
2023,46599,15190,3.07,22423,2078.18
2024,89449,24490,3.65,34008,2630.23
2025,169390,34624,4.89,48149,3518.04


Simulated claims: 332,740


In [44]:
# Create monthly periods for the analysis window
months = pd.date_range(
    start="2021-01-01",
    end="2025-12-01",
    freq="MS"
)

monthly_exposure = []

for month_start in months:
    month_end = month_start + pd.offsets.MonthEnd(0)

    enrolled = (
        (members_clean["enrollment_date"] <= month_end)
        & (
            members_clean["termination_date"].isna()
            | (members_clean["termination_date"] >= month_start)
        )
    )

    monthly_exposure.append({
        "service_month": month_start,
        "enrolled_members": enrolled.sum()
    })

monthly_exposure = pd.DataFrame(monthly_exposure)

display(monthly_exposure.head(12))
display(monthly_exposure.tail(12))

print(
    "Total member-month exposure: "
    f"{monthly_exposure['enrolled_members'].sum():,}"
)

,service_month,enrolled_members
0,2021-01-01,504
1,2021-02-01,965
2,2021-03-01,1445
3,2021-04-01,1936
4,2021-05-01,2427
5,2021-06-01,2902
6,2021-07-01,3384
7,2021-08-01,3862
8,2021-09-01,4324
9,2021-10-01,4818


,service_month,enrolled_members
48,2025-01-01,34281
49,2025-02-01,35303
50,2025-03-01,36513
51,2025-04-01,37576
52,2025-05-01,38664
53,2025-06-01,39735
54,2025-07-01,40780
55,2025-08-01,41850
56,2025-09-01,42820
57,2025-10-01,43903


Total member-month exposure: 1,185,556


In [45]:
# Calculate overall claim utilization rate per member-month
overall_utilization_rate = (
    len(claims_clean)
    / monthly_exposure["enrolled_members"].sum()
)

print(
    f"Overall claims per 1,000 member-months: "
    f"{overall_utilization_rate * 1000:.2f}"
)

# Calculate expected claim volume based on enrollment exposure
monthly_exposure["expected_claim_lines"] = (
    monthly_exposure["enrolled_members"]
    * overall_utilization_rate
)

display(
    monthly_exposure[
        [
            "service_month",
            "enrolled_members",
            "expected_claim_lines"
        ]
    ].head(12).round(2)
)

display(
    monthly_exposure[
        [
            "service_month",
            "enrolled_members",
            "expected_claim_lines"
        ]
    ].tail(12).round(2)
)

print(
    "Expected claims across all months: "
    f"{monthly_exposure['expected_claim_lines'].sum():,.0f}"
)

Overall claims per 1,000 member-months: 280.66


,service_month,enrolled_members,expected_claim_lines
0,2021-01-01,504,141.45
1,2021-02-01,965,270.84
2,2021-03-01,1445,405.56
3,2021-04-01,1936,543.36
4,2021-05-01,2427,681.17
5,2021-06-01,2902,814.48
6,2021-07-01,3384,949.76
7,2021-08-01,3862,1083.91
8,2021-09-01,4324,1213.58
9,2021-10-01,4818,1352.23


,service_month,enrolled_members,expected_claim_lines
48,2025-01-01,34281,9621.36
49,2025-02-01,35303,9908.20
50,2025-03-01,36513,10247.80
51,2025-04-01,37576,10546.14
52,2025-05-01,38664,10851.50
53,2025-06-01,39735,11152.09
54,2025-07-01,40780,11445.38
55,2025-08-01,41850,11745.69
56,2025-09-01,42820,12017.93
57,2025-10-01,43903,12321.88


Expected claims across all months: 332,740


In [46]:
# Set seed for reproducible monthly variation
rng = np.random.default_rng(42)

# Generate controlled monthly utilization variation
monthly_variation = rng.normal(
    loc=1.0,
    scale=0.04,
    size=len(monthly_exposure)
)

# Limit variation to approximately ±10%
monthly_variation = np.clip(
    monthly_variation,
    0.90,
    1.10
)

monthly_exposure["variation_factor"] = monthly_variation

# Apply variation to exposure-based expected volume
monthly_exposure["adjusted_expected_claims"] = (
    monthly_exposure["expected_claim_lines"]
    * monthly_exposure["variation_factor"]
)

# Normalize so total claim volume remains unchanged
normalization_factor = (
    len(claims_clean)
    / monthly_exposure["adjusted_expected_claims"].sum()
)

monthly_exposure["target_claim_lines"] = (
    monthly_exposure["adjusted_expected_claims"]
    * normalization_factor
).round().astype(int)

# Correct rounding difference so targets equal the exact claim total
rounding_difference = (
    len(claims_clean)
    - monthly_exposure["target_claim_lines"].sum()
)

monthly_exposure.loc[
    monthly_exposure.index[-1],
    "target_claim_lines"
] += rounding_difference

# Calculate target utilization rate
monthly_exposure["target_claims_per_1000"] = (
    monthly_exposure["target_claim_lines"]
    / monthly_exposure["enrolled_members"]
    * 1000
)

display(
    monthly_exposure[
        [
            "service_month",
            "enrolled_members",
            "target_claim_lines",
            "target_claims_per_1000"
        ]
    ].head(12).round(2)
)

display(
    monthly_exposure[
        [
            "service_month",
            "enrolled_members",
            "target_claim_lines",
            "target_claims_per_1000"
        ]
    ].tail(12).round(2)
)

print(
    f"Target claims: "
    f"{monthly_exposure['target_claim_lines'].sum():,}"
)

print(
    "Target utilization range: "
    f"{monthly_exposure['target_claims_per_1000'].min():.2f} to "
    f"{monthly_exposure['target_claims_per_1000'].max():.2f}"
)

,service_month,enrolled_members,target_claim_lines,target_claims_per_1000
0,2021-01-01,504,143,283.73
1,2021-02-01,965,259,268.39
2,2021-03-01,1445,416,287.89
3,2021-04-01,1936,562,290.29
4,2021-05-01,2427,626,257.93
5,2021-06-01,2902,769,264.99
6,2021-07-01,3384,951,281.03
7,2021-08-01,3862,1066,276.02
8,2021-09-01,4324,1208,279.37
9,2021-10-01,4818,1301,270.03


,service_month,enrolled_members,target_claim_lines,target_claims_per_1000
48,2025-01-01,34281,9845,287.19
49,2025-02-01,35303,9897,280.34
50,2025-03-01,36513,10327,282.83
51,2025-04-01,37576,10771,286.65
52,2025-05-01,38664,10180,263.29
53,2025-06-01,39735,10967,276.00
54,2025-07-01,40780,11187,274.33
55,2025-08-01,41850,11402,272.45
56,2025-09-01,42820,11840,276.51
57,2025-10-01,43903,13009,296.31


Target claims: 332,740
Target utilization range: 257.93 to 303.55


In [47]:
# Check whether enough eligible claims exist for each monthly target
feasibility_results = []

for _, row in monthly_exposure.iterrows():

    month_start = row["service_month"]
    month_end = month_start + pd.offsets.MonthEnd(0)
    target = row["target_claim_lines"]

    eligible = (
        (dq013["enrollment_date"] <= month_end)
        & (dq013["valid_end_date"] >= month_start)
    )

    eligible_claims = eligible.sum()

    feasibility_results.append({
        "service_month": month_start,
        "target_claim_lines": target,
        "eligible_claims": eligible_claims,
        "target_feasible": eligible_claims >= target
    })

dq013_feasibility = pd.DataFrame(feasibility_results)

display(dq013_feasibility.head(12))
display(dq013_feasibility.tail(12))

print(
    "Months without enough eligible claims: "
    f"{(~dq013_feasibility['target_feasible']).sum():,}"
)

print(
    "Minimum eligible claim surplus: "
    f"{(
        dq013_feasibility['eligible_claims']
        - dq013_feasibility['target_claim_lines']
    ).min():,}"
)

,service_month,target_claim_lines,eligible_claims,target_feasible
0,2021-01-01,143,4325,True
1,2021-02-01,259,8477,True
2,2021-03-01,416,12837,True
3,2021-04-01,562,17134,True
4,2021-05-01,626,21675,True
5,2021-06-01,769,25998,True
6,2021-07-01,951,30867,True
7,2021-08-01,1066,35593,True
8,2021-09-01,1208,40051,True
9,2021-10-01,1301,44543,True


,service_month,target_claim_lines,eligible_claims,target_feasible
48,2025-01-01,9845,285268,True
49,2025-02-01,9897,291252,True
50,2025-03-01,10327,297285,True
51,2025-04-01,10771,302360,True
52,2025-05-01,10180,307539,True
53,2025-06-01,10967,312205,True
54,2025-07-01,11187,315747,True
55,2025-08-01,11402,318749,True
56,2025-09-01,11840,321239,True
57,2025-10-01,13009,323050,True


Months without enough eligible claims: 0
Minimum eligible claim surplus: 4,182


In [48]:
# Create a fresh assignment workspace
dq013_assignment = dq013.copy()

dq013_assignment["assigned_month"] = pd.NaT

# Set seed for reproducibility
rng = np.random.default_rng(42)

assignment_failed = False

# Assign claims from latest month to earliest month
for _, row in monthly_exposure.iloc[::-1].iterrows():

    month_start = row["service_month"]
    month_end = month_start + pd.offsets.MonthEnd(0)
    target = int(row["target_claim_lines"])

    # Claim must be unassigned and member must be enrolled
    eligible_mask = (
        dq013_assignment["assigned_month"].isna()
        & (dq013_assignment["enrollment_date"] <= month_end)
        & (dq013_assignment["valid_end_date"] >= month_start)
    )

    eligible_indices = (
        dq013_assignment.index[eligible_mask]
        .to_numpy()
    )

    if len(eligible_indices) < target:
        print(
            f"Assignment failed for {month_start:%Y-%m}: "
            f"target={target:,}, "
            f"eligible={len(eligible_indices):,}"
        )

        assignment_failed = True
        break

    selected_indices = rng.choice(
        eligible_indices,
        size=target,
        replace=False
    )

    dq013_assignment.loc[
        selected_indices,
        "assigned_month"
    ] = month_start

# Check assignment results
assigned_claims = (
    dq013_assignment["assigned_month"]
    .notna()
    .sum()
)

unassigned_claims = (
    dq013_assignment["assigned_month"]
    .isna()
    .sum()
)

print(f"Assignment failed: {assignment_failed}")
print(f"Assigned claims:   {assigned_claims:,}")
print(f"Unassigned claims: {unassigned_claims:,}")

Assignment failed for 2023-06: target=4,944, eligible=143
Assignment failed: True
Assigned claims:   263,609
Unassigned claims: 69,131


In [49]:
# Build cumulative feasibility check
cumulative_check = []

for _, row in monthly_exposure.iterrows():

    month_start = row["service_month"]
    month_end = month_start + pd.offsets.MonthEnd(0)

    cumulative_target = (
        monthly_exposure.loc[
            monthly_exposure["service_month"] <= month_start,
            "target_claim_lines"
        ].sum()
    )

    # Claims whose enrollment window has ended by this month
    must_be_assigned = (
        dq013["valid_end_date"] <= month_end
    ).sum()

    # Claims whose enrollment has begun by this month
    available_by_month = (
        dq013["enrollment_date"] <= month_end
    ).sum()

    cumulative_check.append({
        "service_month": month_start,
        "must_be_assigned": must_be_assigned,
        "cumulative_target": cumulative_target,
        "available_by_month": available_by_month,
        "lower_bound_ok": cumulative_target >= must_be_assigned,
        "upper_bound_ok": cumulative_target <= available_by_month
    })

dq013_cumulative_check = pd.DataFrame(
    cumulative_check
)

display(
    dq013_cumulative_check[
        [
            "service_month",
            "must_be_assigned",
            "cumulative_target",
            "available_by_month",
            "lower_bound_ok",
            "upper_bound_ok"
        ]
    ]
)

print(
    "Lower-bound violations: "
    f"{(~dq013_cumulative_check['lower_bound_ok']).sum():,}"
)

print(
    "Upper-bound violations: "
    f"{(~dq013_cumulative_check['upper_bound_ok']).sum():,}"
)

,service_month,must_be_assigned,cumulative_target,available_by_month,lower_bound_ok,upper_bound_ok
0,2021-01-01,0,143,4325,True,True
1,2021-02-01,0,402,8477,True,True
2,2021-03-01,0,818,12837,True,True
3,2021-04-01,0,1380,17134,True,True
4,2021-05-01,0,2006,21675,True,True
5,2021-06-01,0,2775,25998,True,True
6,2021-07-01,0,3726,30867,True,True
7,2021-08-01,0,4792,35593,True,True
8,2021-09-01,0,6000,40051,True,True
9,2021-10-01,0,7301,44543,True,True


Lower-bound violations: 0
Upper-bound violations: 0


In [50]:
# Inspect claim-level structure before redistributing service dates
claim_structure = (
    claims_clean
    .groupby("claim_id")
    .agg(
        service_lines=("service_line_number", "size"),
        unique_members=("member_id", "nunique"),
        unique_service_dates=("service_date", "nunique")
    )
)

print(
    f"Unique claims: "
    f"{len(claim_structure):,}"
)

print(
    f"Multi-line claims: "
    f"{(claim_structure['service_lines'] > 1).sum():,}"
)

print(
    f"Maximum service lines on one claim: "
    f"{claim_structure['service_lines'].max():,}"
)

print(
    "Claims associated with multiple members: "
    f"{(claim_structure['unique_members'] > 1).sum():,}"
)

print(
    "Claims currently containing multiple service dates: "
    f"{(claim_structure['unique_service_dates'] > 1).sum():,}"
)

Unique claims: 207,653
Multi-line claims: 83,085
Maximum service lines on one claim: 4
Claims associated with multiple members: 0
Claims currently containing multiple service dates: 0


In [51]:
# Create one record per claim for temporal redistribution
dq013_claims = (
    dq013
    .groupby("claim_id")
    .agg(
        member_id=("member_id", "first"),
        service_lines=("service_line_number", "size"),
        enrollment_date=("enrollment_date", "first"),
        valid_end_date=("valid_end_date", "first"),
        original_service_date=("service_date", "first")
    )
    .reset_index()
)

print(f"Claim-level records: {len(dq013_claims):,}")

print(
    "Service lines represented: "
    f"{dq013_claims['service_lines'].sum():,}"
)

print(
    "Claims with invalid enrollment windows: "
    f"{(
        dq013_claims['enrollment_date']
        > dq013_claims['valid_end_date']
    ).sum():,}"
)

display(dq013_claims.head())

Claim-level records: 207,653
Service lines represented: 332,740
Claims with invalid enrollment windows: 0


,claim_id,member_id,service_lines,enrollment_date,valid_end_date,original_service_date
0,CLM00000001,MBR001379,1,2021-12-04,2025-12-31,2023-10-15
1,CLM00000006,MBR017080,1,2024-09-10,2025-12-31,2025-07-29
2,CLM00000008,MBR002416,1,2023-07-22,2025-12-31,2025-08-26
3,CLM00000009,MBR037993,4,2022-05-14,2025-12-31,2025-11-26
4,CLM00000010,MBR036110,1,2025-02-15,2025-12-31,2025-10-18


In [52]:
# Calculate the first and last eligible month for each claim
dq013_claims["first_eligible_month"] = (
    dq013_claims["enrollment_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

dq013_claims["last_eligible_month"] = (
    dq013_claims["valid_end_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Calculate number of eligible months for each claim
dq013_claims["eligible_months"] = (
    (
        dq013_claims["last_eligible_month"].dt.year
        - dq013_claims["first_eligible_month"].dt.year
    ) * 12
    + (
        dq013_claims["last_eligible_month"].dt.month
        - dq013_claims["first_eligible_month"].dt.month
    )
    + 1
)

print(
    "Minimum eligible months: "
    f"{dq013_claims['eligible_months'].min():,}"
)

print(
    "Maximum eligible months: "
    f"{dq013_claims['eligible_months'].max():,}"
)

print(
    "Median eligible months: "
    f"{dq013_claims['eligible_months'].median():.0f}"
)

print(
    "Claims eligible for only one month: "
    f"{(dq013_claims['eligible_months'] == 1).sum():,}"
)

display(
    dq013_claims[
        [
            "claim_id",
            "member_id",
            "service_lines",
            "first_eligible_month",
            "last_eligible_month",
            "eligible_months"
        ]
    ]
    .sort_values("eligible_months")
    .head(10)
)

Minimum eligible months: 1
Maximum eligible months: 60
Median eligible months: 29
Claims eligible for only one month: 429


,claim_id,member_id,service_lines,first_eligible_month,last_eligible_month,eligible_months
20118,CLM00029937,MBR029109,4,2025-12-01,2025-12-01,1
54077,CLM00081035,MBR034742,2,2025-12-01,2025-12-01,1
69313,CLM00103891,MBR046712,1,2025-12-01,2025-12-01,1
142528,CLM00213835,MBR003101,1,2025-12-01,2025-12-01,1
181346,CLM00272513,MBR023939,1,2025-12-01,2025-12-01,1
6620,CLM00009828,MBR031770,1,2025-12-01,2025-12-01,1
105213,CLM00157710,MBR020479,3,2025-12-01,2025-12-01,1
95275,CLM00142764,MBR014849,2,2025-12-01,2025-12-01,1
72917,CLM00109247,MBR038325,1,2025-12-01,2025-12-01,1
105159,CLM00157622,MBR024040,1,2025-12-01,2025-12-01,1


In [53]:
# Create fresh claim-level assignment workspace
dq013_claim_assignment = dq013_claims.copy()

dq013_claim_assignment["assigned_month"] = pd.NaT

# Store remaining service-line capacity for each month
remaining_capacity = dict(
    zip(
        monthly_exposure["service_month"],
        monthly_exposure["target_claim_lines"]
    )
)

# Process the most constrained claims first
assignment_order = (
    dq013_claim_assignment
    .sort_values(
        ["eligible_months", "service_lines"],
        ascending=[True, False]
    )
    .index
)

unassigned_claim_ids = []

for idx in assignment_order:

    claim = dq013_claim_assignment.loc[idx]

    first_month = claim["first_eligible_month"]
    last_month = claim["last_eligible_month"]
    service_lines = int(claim["service_lines"])

    # Find eligible months with enough remaining capacity
    eligible_months = [
        month
        for month in monthly_exposure["service_month"]
        if (
            first_month <= month <= last_month
            and remaining_capacity[month] >= service_lines
        )
    ]

    if not eligible_months:
        unassigned_claim_ids.append(claim["claim_id"])
        continue

    # Choose eligible month with the greatest remaining capacity
    assigned_month = max(
        eligible_months,
        key=lambda month: remaining_capacity[month]
    )

    dq013_claim_assignment.loc[
        idx,
        "assigned_month"
    ] = assigned_month

    remaining_capacity[assigned_month] -= service_lines


# Summarize results
unassigned_claims = (
    dq013_claim_assignment["assigned_month"]
    .isna()
    .sum()
)

unassigned_service_lines = (
    dq013_claim_assignment.loc[
        dq013_claim_assignment["assigned_month"].isna(),
        "service_lines"
    ].sum()
)

remaining_service_line_capacity = sum(
    remaining_capacity.values()
)

print(
    f"Assigned claims: "
    f"{dq013_claim_assignment['assigned_month'].notna().sum():,}"
)

print(
    f"Unassigned claims: "
    f"{unassigned_claims:,}"
)

print(
    f"Unassigned service lines: "
    f"{unassigned_service_lines:,}"
)

print(
    f"Remaining monthly capacity: "
    f"{remaining_service_line_capacity:,}"
)

Assigned claims: 207,653
Unassigned claims: 0
Unassigned service lines: 0
Remaining monthly capacity: 0


In [54]:
# Summarize assigned service lines by month
assigned_monthly = (
    dq013_claim_assignment
    .groupby("assigned_month")
    .agg(
        assigned_claims=("claim_id", "size"),
        assigned_service_lines=("service_lines", "sum")
    )
    .reset_index()
)

# Compare assignments with monthly targets
assignment_validation = (
    monthly_exposure[
        [
            "service_month",
            "target_claim_lines",
            "target_claims_per_1000"
        ]
    ]
    .merge(
        assigned_monthly,
        left_on="service_month",
        right_on="assigned_month",
        how="left"
    )
)

assignment_validation["line_difference"] = (
    assignment_validation["assigned_service_lines"]
    - assignment_validation["target_claim_lines"]
)

display(assignment_validation)

print(
    "Months not matching target: "
    f"{(assignment_validation['line_difference'] != 0).sum():,}"
)

print(
    "Maximum absolute target difference: "
    f"{assignment_validation['line_difference'].abs().max():,}"
)

# Validate assigned month falls within each claim's enrollment window
invalid_month_assignments = (
    (dq013_claim_assignment["assigned_month"]
        < dq013_claim_assignment["first_eligible_month"])
    |
    (dq013_claim_assignment["assigned_month"]
        > dq013_claim_assignment["last_eligible_month"])
).sum()

print(
    "Claims assigned outside eligible months: "
    f"{invalid_month_assignments:,}"
)

,service_month,target_claim_lines,target_claims_per_1000,assigned_month,assigned_claims,assigned_service_lines,line_difference
0,2021-01-01,143,283.730159,2021-01-01,66,143,0
1,2021-02-01,259,268.393782,2021-02-01,116,259,0
2,2021-03-01,416,287.889273,2021-03-01,185,416,0
3,2021-04-01,562,290.289256,2021-04-01,258,562,0
4,2021-05-01,626,257.931603,2021-05-01,298,626,0
5,2021-06-01,769,264.989662,2021-06-01,375,769,0
6,2021-07-01,951,281.028369,2021-07-01,441,951,0
7,2021-08-01,1066,276.022786,2021-08-01,508,1066,0
8,2021-09-01,1208,279.370953,2021-09-01,588,1208,0
9,2021-10-01,1301,270.029058,2021-10-01,642,1301,0


Months not matching target: 0
Maximum absolute target difference: 0
Claims assigned outside eligible months: 0


In [55]:
# Create the first and last calendar day of the assigned month
dq013_claim_assignment["month_start"] = (
    dq013_claim_assignment["assigned_month"]
)

dq013_claim_assignment["month_end"] = (
    dq013_claim_assignment["assigned_month"]
    + pd.offsets.MonthEnd(0)
)

# Restrict possible service dates to the member's enrollment window
dq013_claim_assignment["date_start"] = (
    dq013_claim_assignment[
        ["month_start", "enrollment_date"]
    ].max(axis=1)
)

dq013_claim_assignment["date_end"] = (
    dq013_claim_assignment[
        ["month_end", "valid_end_date"]
    ].min(axis=1)
)

# Confirm every claim has at least one valid day
invalid_date_ranges = (
    dq013_claim_assignment["date_start"]
    > dq013_claim_assignment["date_end"]
).sum()

print(
    "Claims without a valid service-date range: "
    f"{invalid_date_ranges:,}"
)

# Generate reproducible random service dates
rng = np.random.default_rng(42)

valid_days = (
    dq013_claim_assignment["date_end"]
    - dq013_claim_assignment["date_start"]
).dt.days

random_offsets = (
    rng.random(len(dq013_claim_assignment))
    * (valid_days + 1)
).astype(int)

dq013_claim_assignment["new_service_date"] = (
    dq013_claim_assignment["date_start"]
    + pd.to_timedelta(random_offsets, unit="D")
)

# Validate generated dates
before_enrollment = (
    dq013_claim_assignment["new_service_date"]
    < dq013_claim_assignment["enrollment_date"]
).sum()

after_valid_end = (
    dq013_claim_assignment["new_service_date"]
    > dq013_claim_assignment["valid_end_date"]
).sum()

wrong_month = (
    dq013_claim_assignment["new_service_date"]
        .dt.to_period("M")
    != dq013_claim_assignment["assigned_month"]
        .dt.to_period("M")
).sum()

print(
    f"Dates before enrollment: {before_enrollment:,}"
)

print(
    f"Dates after valid enrollment end: {after_valid_end:,}"
)

print(
    f"Dates outside assigned month: {wrong_month:,}"
)

display(
    dq013_claim_assignment[
        [
            "claim_id",
            "member_id",
            "assigned_month",
            "date_start",
            "date_end",
            "new_service_date"
        ]
    ].head(10)
)


Claims without a valid service-date range: 0
Dates before enrollment: 0
Dates after valid enrollment end: 0
Dates outside assigned month: 0


,claim_id,member_id,assigned_month,date_start,date_end,new_service_date
0,CLM00000001,MBR001379,2023-04-01,2023-04-01,2023-04-30,2023-04-24
1,CLM00000006,MBR017080,2024-09-01,2024-09-10,2024-09-30,2024-09-19
2,CLM00000008,MBR002416,2025-11-01,2025-11-01,2025-11-30,2025-11-26
3,CLM00000009,MBR037993,2022-05-01,2022-05-14,2022-05-31,2022-05-26
4,CLM00000010,MBR036110,2025-11-01,2025-11-01,2025-11-30,2025-11-03
5,CLM00000011,MBR013213,2025-04-01,2025-04-01,2025-04-30,2025-04-30
6,CLM00000012,MBR036992,2025-07-01,2025-07-01,2025-07-31,2025-07-24
7,CLM00000013,MBR020709,2025-07-01,2025-07-01,2025-07-31,2025-07-25
8,CLM00000015,MBR005570,2025-04-01,2025-04-01,2025-04-30,2025-04-04
9,CLM00000016,MBR033636,2025-08-01,2025-08-01,2025-08-31,2025-08-14


In [56]:
# Preserve service dates before DQ-013 cleaning
claims_clean["service_date_original"] = (
    claims_clean["service_date"]
)

# Create claim-level service date mapping
dq013_date_map = (
    dq013_claim_assignment[
        ["claim_id", "new_service_date"]
    ]
    .set_index("claim_id")["new_service_date"]
)

# Apply corrected service date to every service line
claims_clean["service_date"] = (
    claims_clean["claim_id"]
    .map(dq013_date_map)
)

print(
    "Claims missing corrected service date: "
    f"{claims_clean['service_date'].isna().sum():,}"
)

print(
    "Service lines retained: "
    f"{len(claims_clean):,}"
)

print(
    "Unique claims retained: "
    f"{claims_clean['claim_id'].nunique():,}"
)

Claims missing corrected service date: 0
Service lines retained: 332,740
Unique claims retained: 207,653


### Validation

Validate that the temporal redistribution preserved claim structure and enrollment eligibility before evaluating the corrected utilization trend.

In [57]:
# Join corrected claims to member enrollment dates
dq013_validation = claims_clean.merge(
    members_clean[
        [
            "member_id",
            "enrollment_date",
            "termination_date"
        ]
    ],
    on="member_id",
    how="left"
)

# Validate enrollment eligibility
before_enrollment = (
    dq013_validation["service_date"]
    < dq013_validation["enrollment_date"]
).sum()

after_termination = (
    dq013_validation["termination_date"].notna()
    & (
        dq013_validation["service_date"]
        > dq013_validation["termination_date"]
    )
).sum()

# Validate all service lines on a claim retain one service date
claim_date_validation = (
    claims_clean
    .groupby("claim_id")["service_date"]
    .nunique()
)

multiple_dates = (
    claim_date_validation > 1
).sum()

# Confirm record structure was preserved
print(f"Service lines retained: {len(claims_clean):,}")
print(
    f"Unique claims retained: "
    f"{claims_clean['claim_id'].nunique():,}"
)
print(f"Claims before enrollment: {before_enrollment:,}")
print(f"Claims after termination: {after_termination:,}")
print(
    "Claims containing multiple service dates: "
    f"{multiple_dates:,}"
)

Service lines retained: 332,740
Unique claims retained: 207,653
Claims before enrollment: 0
Claims after termination: 0
Claims containing multiple service dates: 0


In [58]:
# Calculate corrected monthly claim-line volume
corrected_monthly = (
    claims_clean
    .assign(
        service_month=(
            claims_clean["service_date"]
            .dt.to_period("M")
            .dt.to_timestamp()
        )
    )
    .groupby("service_month")
    .size()
    .reset_index(name="claim_lines")
)

# Join monthly enrollment exposure
corrected_monthly = corrected_monthly.merge(
    monthly_exposure[
        [
            "service_month",
            "enrolled_members"
        ]
    ],
    on="service_month",
    how="left"
)

# Calculate utilization per 1,000 enrolled members
corrected_monthly["claims_per_1000"] = (
    corrected_monthly["claim_lines"]
    / corrected_monthly["enrolled_members"]
    * 1000
)

display(corrected_monthly.head(12).round(2))
display(corrected_monthly.tail(12).round(2))

print(
    "Corrected utilization range: "
    f"{corrected_monthly['claims_per_1000'].min():.2f} to "
    f"{corrected_monthly['claims_per_1000'].max():.2f}"
)

print(
    "Average monthly utilization: "
    f"{corrected_monthly['claims_per_1000'].mean():.2f}"
)

print(
    "January 2021 utilization: "
    f"{corrected_monthly.iloc[0]['claims_per_1000']:.2f}"
)

print(
    "December 2025 utilization: "
    f"{corrected_monthly.iloc[-1]['claims_per_1000']:.2f}"
)

,service_month,claim_lines,enrolled_members,claims_per_1000
0,2021-01-01,143,504,283.73
1,2021-02-01,259,965,268.39
2,2021-03-01,416,1445,287.89
3,2021-04-01,562,1936,290.29
4,2021-05-01,626,2427,257.93
5,2021-06-01,769,2902,264.99
6,2021-07-01,951,3384,281.03
7,2021-08-01,1066,3862,276.02
8,2021-09-01,1208,4324,279.37
9,2021-10-01,1301,4818,270.03


,service_month,claim_lines,enrolled_members,claims_per_1000
48,2025-01-01,9845,34281,287.19
49,2025-02-01,9897,35303,280.34
50,2025-03-01,10327,36513,282.83
51,2025-04-01,10771,37576,286.65
52,2025-05-01,10180,38664,263.29
53,2025-06-01,10967,39735,276.00
54,2025-07-01,11187,40780,274.33
55,2025-08-01,11402,41850,272.45
56,2025-09-01,11840,42820,276.51
57,2025-10-01,13009,43903,296.31


Corrected utilization range: 257.93 to 303.55
Average monthly utilization: 280.34
January 2021 utilization: 283.73
December 2025 utilization: 290.37


### Interpretation

Validation confirmed that the temporal redistribution removed the artificial year-over-year increase in utilization intensity while preserving realistic month-to-month variation.

After controlling for enrollment, monthly utilization ranged from 257.93 to 303.55 claim lines per 1,000 enrolled members, with an average of 280.34. Utilization was 283.73 per 1,000 in January 2021 and 290.37 in December 2025, indicating that the previous sustained temporal growth pattern was removed.

Total claim volume continues to increase over time as member enrollment grows, while utilization intensity remains relatively stable. All 332,740 service lines and 207,653 claims were retained, no corrected service dates fell outside member enrollment periods, and all service lines belonging to the same claim retained a consistent service date.

## Final Cross-Dataset Validation

Perform final validation of the cleaned Claims, Members, and Providers datasets after all approved cleaning actions. Confirm dataset structure, key integrity, referential integrity, business-rule compliance, and analytical consistency before freezing the analysis-ready datasets.

### Structural Integrity

Confirm that the cleaned datasets retain their intended grain, required identifiers, and unique keys.

In [59]:
# Confirm cleaned dataset dimensions
print(f"Claims:    {claims_clean.shape}")
print(f"Members:   {members_clean.shape}")
print(f"Providers: {providers_clean.shape}")

print()

# Validate primary and composite keys
print(
    "Duplicate Member IDs: "
    f"{members_clean['member_id'].duplicated().sum():,}"
)

print(
    "Missing Member IDs: "
    f"{members_clean['member_id'].isna().sum():,}"
)

print(
    "Duplicate Provider IDs: "
    f"{providers_clean['provider_id'].duplicated().sum():,}"
)

print(
    "Missing Provider IDs: "
    f"{providers_clean['provider_id'].isna().sum():,}"
)

duplicate_claim_lines = (
    claims_clean
    .duplicated(
        subset=["claim_id", "service_line_number"]
    )
    .sum()
)

print(
    "Duplicate Claim ID + Service Line: "
    f"{duplicate_claim_lines:,}"
)

print(
    "Missing Claim IDs: "
    f"{claims_clean['claim_id'].isna().sum():,}"
)

print(
    "Missing Service Line Numbers: "
    f"{claims_clean['service_line_number'].isna().sum():,}"
)

Claims:    (332740, 17)
Members:   (50000, 8)
Providers: (2500, 6)

Duplicate Member IDs: 0
Missing Member IDs: 0
Duplicate Provider IDs: 0
Missing Provider IDs: 0
Duplicate Claim ID + Service Line: 0
Missing Claim IDs: 0
Missing Service Line Numbers: 0


### Referential Integrity

Confirm that every member and provider referenced by the cleaned claims dataset exists in the corresponding dimension table after all cleaning and provider reassignment activities.

In [60]:
# Identify claim member IDs not found in Members
orphan_members = (
    ~claims_clean["member_id"]
    .isin(members_clean["member_id"])
).sum()

# Identify claim provider IDs not found in Providers
orphan_providers = (
    ~claims_clean["provider_id"]
    .isin(providers_clean["provider_id"])
).sum()

print(
    "Claims with invalid Member ID: "
    f"{orphan_members:,}"
)

print(
    "Claims with invalid Provider ID: "
    f"{orphan_providers:,}"
)

Claims with invalid Member ID: 0
Claims with invalid Provider ID: 0


### Business Rule Validation

Revalidate the major business rules affected by cleaning to confirm that corrections remain valid when the cleaned datasets are evaluated together.

#### Enrollment and Claim Eligibility

Confirm valid member enrollment chronology and verify that all retained claim service dates fall within the corresponding member enrollment period.

In [61]:
# Validate member enrollment chronology
invalid_member_enrollment = (
    members_clean["enrollment_date"]
    < members_clean["date_of_birth"]
).sum()

# Join claims to member enrollment information
eligibility_validation = claims_clean.merge(
    members_clean[
        [
            "member_id",
            "enrollment_date",
            "termination_date"
        ]
    ],
    on="member_id",
    how="left"
)

# Validate claim service dates
claims_before_enrollment = (
    eligibility_validation["service_date"]
    < eligibility_validation["enrollment_date"]
).sum()

claims_after_termination = (
    eligibility_validation["termination_date"].notna()
    & (
        eligibility_validation["service_date"]
        > eligibility_validation["termination_date"]
    )
).sum()

print(
    "Members enrolled before DOB: "
    f"{invalid_member_enrollment:,}"
)

print(
    "Claims before enrollment: "
    f"{claims_before_enrollment:,}"
)

print(
    "Claims after termination: "
    f"{claims_after_termination:,}"
)

Members enrolled before DOB: 0
Claims before enrollment: 0
Claims after termination: 0


### Provider Compatibility and Geography

Confirm that all claim provider assignments comply with the documented service category, place-of-service, and provider-type rules. Revalidate the geographic distribution after provider reassignment to ensure that same-state service remains reasonably aligned with the intended approximate target while retaining realistic cross-state care.

In [62]:
# Join final provider and member attributes to claims
provider_validation = (
    claims_clean
    .merge(
        members_clean[
            ["member_id", "state"]
        ].rename(
            columns={"state": "member_state"}
        ),
        on="member_id",
        how="left"
    )
    .merge(
        providers_clean[
            ["provider_id", "provider_type", "state"]
        ].rename(
            columns={"state": "provider_state"}
        ),
        on="provider_id",
        how="left"
    )
)

# Validate provider compatibility
def is_provider_compatible(row):
    valid_provider_types = dq011_pos_rules.get(
        (
            row["service_category"],
            row["place_of_service"]
        ),
        []
    )

    return row["provider_type"] in valid_provider_types


provider_validation["provider_compatible"] = (
    provider_validation.apply(
        is_provider_compatible,
        axis=1
    )
)

compatibility_violations = (
    ~provider_validation["provider_compatible"]
).sum()

# Validate geography
provider_validation["geography"] = np.where(
    provider_validation["member_state"]
    == provider_validation["provider_state"],
    "Same State",
    "Cross State"
)

geography_validation = (
    provider_validation["geography"]
    .value_counts()
    .rename_axis("Geography")
    .reset_index(name="Claim Lines")
)

geography_validation["Percent"] = (
    geography_validation["Claim Lines"]
    / len(provider_validation)
    * 100
).round(2)

print(
    "Provider compatibility violations: "
    f"{compatibility_violations:,}"
)

print(
    "Claim lines validated: "
    f"{len(provider_validation):,}"
)

display(geography_validation)

Provider compatibility violations: 0
Claim lines validated: 332,740


,Geography,Claim Lines,Percent
0,Same State,310285,93.25
1,Cross State,22455,6.75


### Allowed Amount and Network Effect

Confirm that cleaned allowed amounts remain valid and that the documented out-of-network cost effect is preserved after all provider and temporal corrections.

In [63]:
# Validate allowed amount completeness and validity
missing_allowed_amount = (
    claims_clean["allowed_amount"].isna().sum()
)

invalid_allowed_amount = (
    claims_clean["allowed_amount"] <= 0
).sum()

# Recalculate normalized allowed amount for final validation
cost_validation = (
    claims_clean
    .groupby("network_status")
    .agg(
        claim_lines=("claim_id", "size"),
        avg_allowed_amount=("allowed_amount", "mean"),
        avg_normalized_allowed=(
            "normalized_allowed_amount",
            "mean"
        )
    )
    .round(4)
)

in_network_avg = cost_validation.loc[
    "In-Network",
    "avg_normalized_allowed"
]

out_network_avg = cost_validation.loc[
    "Out-of-Network",
    "avg_normalized_allowed"
]

network_effect = (
    (out_network_avg / in_network_avg) - 1
) * 100

print(
    "Missing allowed amounts: "
    f"{missing_allowed_amount:,}"
)

print(
    "Allowed amounts <= 0: "
    f"{invalid_allowed_amount:,}"
)

display(cost_validation)

print(
    "Out-of-network normalized cost increase: "
    f"{network_effect:.2f}%"
)

Missing allowed amounts: 0
Allowed amounts <= 0: 0


,claim_lines,avg_allowed_amount,avg_normalized_allowed
network_status,,,
In-Network,265419,251.8155,1.0
Out-of-Network,67321,311.2320,1.2


Out-of-network normalized cost increase: 20.00%


### Temporal Utilization

Confirm that the corrected temporal distribution remains stable after all cleaning activities by evaluating monthly claim-line utilization relative to enrolled member exposure.

In [64]:
# Create the 60-month validation period
validation_months = pd.period_range(
    start="2021-01",
    end="2025-12",
    freq="M"
)

# Calculate enrolled members for each month
monthly_exposure = []

for month in validation_months:
    month_start = month.start_time
    month_end = month.end_time.normalize()

    enrolled = (
        (members_clean["enrollment_date"] <= month_end)
        & (
            members_clean["termination_date"].isna()
            | (members_clean["termination_date"] >= month_start)
        )
    ).sum()

    monthly_exposure.append({
        "service_month": month,
        "enrolled_members": enrolled
    })

monthly_exposure = pd.DataFrame(monthly_exposure)

# Calculate cleaned claim lines by month
temporal_claims = (
    claims_clean
    .assign(
        service_month=claims_clean["service_date"].dt.to_period("M")
    )
    .groupby("service_month")
    .size()
    .rename("claim_lines")
    .reset_index()
)

# Combine claims with enrollment exposure
temporal_validation = monthly_exposure.merge(
    temporal_claims,
    on="service_month",
    how="left"
)

temporal_validation["claim_lines"] = (
    temporal_validation["claim_lines"]
    .fillna(0)
    .astype(int)
)

# Calculate exposure-adjusted utilization
temporal_validation["claims_per_1000"] = (
    temporal_validation["claim_lines"]
    / temporal_validation["enrolled_members"]
    * 1000
)

print(
    "Months validated: "
    f"{len(temporal_validation):,}"
)

print(
    "Minimum monthly utilization: "
    f"{temporal_validation['claims_per_1000'].min():.2f}"
)

print(
    "Maximum monthly utilization: "
    f"{temporal_validation['claims_per_1000'].max():.2f}"
)

print(
    "Average monthly utilization: "
    f"{temporal_validation['claims_per_1000'].mean():.2f}"
)

print(
    "January 2021 utilization: "
    f"{temporal_validation.iloc[0]['claims_per_1000']:.2f}"
)

print(
    "December 2025 utilization: "
    f"{temporal_validation.iloc[-1]['claims_per_1000']:.2f}"
)

Months validated: 60
Minimum monthly utilization: 257.93
Maximum monthly utilization: 303.55
Average monthly utilization: 280.34
January 2021 utilization: 283.73
December 2025 utilization: 290.37


### Interpretation

Final cross dataset validation confirmed that the cleaned Healthcare Claims Intelligence data model is structurally consistent and satisfies the documented integrity and business rules.

Primary and composite keys remained unique and complete, and all claim member and provider references were valid. Member enrollment chronology and claim eligibility rules produced zero violations. Provider assignments produced zero compatibility violations, while the final geographic distribution retained realistic cross state care with 93.25% of claim lines occurring within the member's state.

Allowed amounts remained complete and positive, and the documented network cost effect was preserved, with out of network services averaging 20.00% higher normalized allowed amounts than in-network services.

The corrected temporal distribution also remained stable after all cleaning activities. Monthly utilization ranged from 257.93 to 303.55 claim lines per 1,000 enrolled members, with an average of 280.34, confirming that the artificial year over year utilization growth identified during profiling was removed.

The cleaned datasets pass final cross dataset validation and are ready to be prepared as analysis ready data.

## Freeze Analysis Ready Data

Prepare the validated datasets for downstream analysis by removing temporary cleaning and validation fields, confirming the final analytical schemas, and saving separate analysis ready datasets without modifying the original raw source files.

### Final Schema Preparation

Remove temporary fields created during cleaning and validation. Retain only the approved analytical fields defined by the Claims, Members, and Providers data model.

In [67]:
# Temporary fields created during cleaning and validation
claims_helper_columns = [
    "baseline_allowed_amount",
    "allowed_amount_original",
    "network_status",
    "normalized_allowed_amount",
    "service_date_original"
]

# Remove temporary fields from final Claims dataset
claims_final = claims_clean.drop(
    columns=claims_helper_columns
).copy()

# Create final dimension datasets
members_final = members_clean.copy()
providers_final = providers_clean.copy()

# Display final dimensions
print(f"Claims Final:    {claims_final.shape}")
print(f"Members Final:   {members_final.shape}")
print(f"Providers Final: {providers_final.shape}")

print("\nClaims columns:")
print(claims_final.columns.tolist())

print("\nMembers columns:")
print(members_final.columns.tolist())

print("\nProviders columns:")
print(providers_final.columns.tolist())

Claims Final:    (332740, 12)
Members Final:   (50000, 8)
Providers Final: (2500, 6)

Claims columns:
['claim_id', 'service_line_number', 'member_id', 'provider_id', 'service_date', 'procedure_code', 'procedure_description', 'diagnosis_code', 'diagnosis_description', 'place_of_service', 'service_category', 'allowed_amount']

Members columns:
['member_id', 'date_of_birth', 'gender', 'state', 'plan_type', 'enrollment_date', 'enrollment_status', 'termination_date']

Providers columns:
['provider_id', 'provider_name', 'provider_type', 'specialty', 'state', 'network_status']


In [68]:
# ============================================================
# Tableau Dataset Preparation
# ============================================================

# Rename dimension state fields before joining
members_tableau = members_final.rename(
    columns={"state": "member_state"}
).copy()

providers_tableau = providers_final.rename(
    columns={"state": "provider_state"}
).copy()

# Join member attributes to Claims
tableau_final = claims_final.merge(
    members_tableau,
    on="member_id",
    how="left",
    validate="many_to_one"
)

# Join provider attributes to Claims
tableau_final = tableau_final.merge(
    providers_tableau,
    on="provider_id",
    how="left",
    validate="many_to_one"
)

# Validate row count
print(f"Claims Final rows:  {len(claims_final):,}")
print(f"Tableau Final rows: {len(tableau_final):,}")
print(f"Tableau columns:    {tableau_final.shape[1]}")
print(f"Row count preserved: {len(tableau_final) == len(claims_final)}")

print("\nState fields:")
print([
    column for column in tableau_final.columns
    if "state" in column.lower()
])

Claims Final rows:  332,740
Tableau Final rows: 332,740
Tableau columns:    24
Row count preserved: True

State fields:
['member_state', 'provider_state']


## Save Clean Datasets

Save the finalized Claims, Members, and Providers datasets separately from the original raw source files. The cleaned files will serve as the analysis ready source for subsequent insights, recommendations, and dashboard development.

In [69]:
# Define output directory
cleaned_data_path = (
    "/content/drive/MyDrive/Resume 📃/Portfolio/"
    "Healthcare Claims Intelligence/Data/Cleaned"
)

# Create Cleaned folder if it does not already exist
os.makedirs(
    cleaned_data_path,
    exist_ok=True
)

# Save finalized analysis-ready datasets
claims_final.to_csv(
    f"{cleaned_data_path}/claims_clean.csv",
    index=False
)

members_final.to_csv(
    f"{cleaned_data_path}/members_clean.csv",
    index=False
)

providers_final.to_csv(
    f"{cleaned_data_path}/providers_clean.csv",
    index=False
)

# Save joined Tableau-ready dataset
tableau_final.to_csv(
    f"{cleaned_data_path}/healthcare_claims_tableau.csv",
    index=False
)

print("Analysis-ready datasets saved.")
print(f"Claims:    {len(claims_final):,} rows")
print(f"Members:   {len(members_final):,} rows")
print(f"Providers: {len(providers_final):,} rows")
print(f"Tableau:   {len(tableau_final):,} rows")

Analysis-ready datasets saved.
Claims:    332,740 rows
Members:   50,000 rows
Providers: 2,500 rows
Tableau:   332,740 rows
